In [26]:
%pip install torch torchvision numpy tensorboard
%pip install torchinfo matplotlib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [27]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms
from torch.utils.tensorboard import SummaryWriter

for folder in ["data", "checkpoints", "runs"]:
    os.makedirs(folder, exist_ok=True)

torch.manual_seed(0)

In [28]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


In [29]:
#getting the data
transform=transforms.ToTensor()

full_train=datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_set=datasets.MNIST(root="data", train=False,download=True, transform=transform)


train_set,val_set=random_split(
    full_train, [59500,500], generator=torch.Generator().manual_seed(0),
)



In [30]:
#checking with one example
img, label=train_set[0] #MNIST data in binary format (no headers). it is not like CSV 
print(tuple(img.shape), img.dtype) #image is a tensor, label is an int
print(label)



(1, 28, 28) torch.float32
9


In [31]:
#dataloaders

batch_size=128

train_loader=DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader=DataLoader(val_set, batch_size=1000, shuffle=False)
test_loader=DataLoader(test_set, batch_size=1000, shuffle=False)


#checking shape
batch_x,batch_y=next(iter(train_loader))
print(batch_x.shape)
print(batch_y.shape)



torch.Size([128, 1, 28, 28])
torch.Size([128])


In [39]:
#model
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1=nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5, padding=2)
        self.conv2=nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, padding=2)
        self.pool=nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu=nn.ReLU()
        self.flatten=nn.Flatten()
        self.fc1=nn.Linear(64*7*7,1024)
        self.dropout=nn.Dropout(p=0.5)
        self.fc2=nn.Linear(1024,10)

    def forward(self,x):
        x=self.pool(self.relu(self.conv1(x)))
        x=self.pool(self.relu(self.conv2(x)))
        x=self.flatten(x)
        x=self.relu(self.fc1(x))
        x=self.fc2(x)

        return x
    
model=CNN().to(device)
print(model)

CNN(
  (conv1): Conv2d(1, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (conv2): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=3136, out_features=1024, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=1024, out_features=10, bias=True)
)


In [43]:
#checking
from torchinfo import summary
summary(model, input_size=(1,1,28,28)) #first "1" is the batch

Layer (type:depth-idx)                   Output Shape              Param #
CNN                                      [1, 10]                   --
├─Conv2d: 1-1                            [1, 32, 28, 28]           832
├─ReLU: 1-2                              [1, 32, 28, 28]           --
├─MaxPool2d: 1-3                         [1, 32, 14, 14]           --
├─Conv2d: 1-4                            [1, 64, 14, 14]           51,264
├─ReLU: 1-5                              [1, 64, 14, 14]           --
├─MaxPool2d: 1-6                         [1, 64, 7, 7]             --
├─Flatten: 1-7                           [1, 3136]                 --
├─Linear: 1-8                            [1, 1024]                 3,212,288
├─ReLU: 1-9                              [1, 1024]                 --
├─Linear: 1-10                           [1, 10]                   10,250
Total params: 3,274,634
Trainable params: 3,274,634
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 13.92
Input size (MB): 0.00


In [45]:
#loss and optimizer
criterion=nn.CrossEntropyLoss() #loss fn

learning_rate=0.1
optimizer=optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
num_epochs=5
checkpoint_path="./checkpoints/model.pt"
writer=SummaryWriter("./runs/mnist_cnn")


global_step=0
for epoch in range(num_epochs):

    #training
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x,batch_y=batch_x.to(device),batch_y.to(device)

        optimizer.zero_grad() #clear last step's gradients
        #graddient accumualtion is useful for the case where batches don't fit in memory

        logits=model(batch_x) #forward pass
        loss=criterion(logits,batch_y) 
        loss.backward() #backward pass (compute and accumulate gradients)
        optimizer.step() #update weights (gradients)

        writer.add_scalar("loss/train", loss.item(), global_step)
        global_step+=1


    #validating after each epoch
    model.eval()
    correct=0
    total=0

    for batch_x, batch_y in val_loader:
        batch_x,batch_y=batch_x.to(device),batch_y.to(device)

        preds=model(batch_x).argmax(dim=1)
        correct+=(preds==batch_y).sum().item()
        total+=batch_y.size(0)
    
    val_acc=correct/total

    writer.add_scalar("accuracy/val", val_acc, epoch)
    print(f"epoch {epoch+1}/{num_epochs} loss {loss.item():.4f} val acc:{val_acc:.4f}") #.item() extracts numerical value out of a tensor

writer.close()

torch.save(model.state_dict(), checkpoint_path)
print("saved model to:", checkpoint_path)






epoch 1/5 loss 0.0802 val acc:0.9660
epoch 2/5 loss 0.0395 val acc:0.9760
epoch 3/5 loss 0.0443 val acc:0.9780
epoch 4/5 loss 0.0180 val acc:0.9840
epoch 5/5 loss 0.0744 val acc:0.9840
saved model to: ./checkpoints/model.pt


In [49]:
#evaluation
eval_model=CNN().to(device)
eval_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
eval_model.eval()

correct=0
total=0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x,batch_y=batch_x.to(device), batch_y.to(device)
        preds=eval_model(batch_x).argmax(dim=1)
        correct+=(preds==batch_y).sum().item()
        total+=batch_y.size(0)

print(f"test accuracy: {correct/total:.4f}")


test accuracy: 0.9892
